# Spike 04 — PageIndex Reasoning Graph

**Goal:** Feed the OCR'd Markdown into PageIndex and build a reasoning graph (tree structure) over the document.

**Time box:** 1-2 hours

**Question to answer:** Does PageIndex successfully index the bilingual OCR output and return a usable reasoning structure?

**Done when:** Documents are indexed in PageIndex and we can query them without errors.

---

### What is Vector-Less Reasoning RAG?

Traditional RAG:
```
Document → chunk into pieces → embed as vectors → store in vector DB
Query    → embed query       → find nearest chunks → send to LLM
```
Problem: chunking breaks tables and cross-references. Arabic embeddings are weaker than English ones.

PageIndex approach:
```
Document → build reasoning graph (understands structure)
Query    → reasoning traversal over graph → find relevant pages
```
This is why it suits our bilingual, table-heavy urban reports.

### API Key Setup
1. Go to [pageindex.ai](https://pageindex.ai)
2. Click "Try Now" or "Book a Demo"
3. Mention you're in a hackathon — many services give free credits for this
4. Add to `.env`: `PAGEINDEX_API_KEY=your_key_here`

---
### ⚠️ Important
PageIndex's API details aren't fully public. This notebook uses the most likely API shape  
based on their documentation. **Adjust endpoint URLs and payload keys** based on what  
their team sends you after signup.

## Step 1 — Setup

> **SDK note:** The official package exports `PageIndexClient` (not `PageIndex` as in some samples).  
> Confirmed methods: `submit_document`, `is_retrieval_ready`, `get_tree`, `submit_query`, `chat_completions`.

In [ ]:
from pageindex import PageIndexClient
from dotenv import load_dotenv
from pathlib import Path
import os, json, time

load_dotenv(dotenv_path="../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY not found in .env")

client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Paths
AR_DIR   = Path("../ocr_output/arabic")
EN_DIR   = Path("../ocr_output/english")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

ar_files = sorted(AR_DIR.glob("*.md"))
en_files = sorted(EN_DIR.glob("*.md"))

print(f"✅ PageIndexClient ready")
print(f"   English pages : {len(en_files)}")
print(f"   Arabic pages  : {len(ar_files)}")

## Step 2 — Combine Pages into Single Documents

`submit_document()` takes a **file path** — one file per document.  
We combine all OCR pages into two files: one English, one Arabic.  
Each page is separated by a clear heading so PageIndex can distinguish them.

In [ ]:
def combine_pages(md_files: list, output_path: Path, lang: str) -> Path:
    """Merge all page .md files into one document with page separators."""
    parts = []
    for f in md_files:
        content = f.read_text(encoding="utf-8").strip()
        parts.append(f"<!-- PAGE: {f.stem} -->\n\n{content}")
    combined = "\n\n---\n\n".join(parts)
    output_path.write_text(combined, encoding="utf-8")
    size_kb = output_path.stat().st_size // 1024
    print(f"✅ {lang}: {len(md_files)} pages → {output_path.name} ({size_kb} KB)")
    return output_path

en_combined = combine_pages(en_files, DATA_DIR / "tranquil_en_combined.md", "English")
ar_combined = combine_pages(ar_files, DATA_DIR / "tranquil_ar_combined.md", "Arabic")

## Step 3 — Submit Documents to PageIndex

`submit_document(file_path)` uploads the file and starts building the reasoning tree.  
It returns immediately with a `doc_id` — indexing continues in the background.

In [ ]:
DOC_ID_CACHE = DATA_DIR / "doc_ids.json"

if DOC_ID_CACHE.exists():
    doc_ids = json.loads(DOC_ID_CACHE.read_text())
    print(f"⏭  Loaded doc IDs from cache:")
    for k, v in doc_ids.items():
        print(f"   {k}: {v}")
else:
    doc_ids = {}
    for label, file_path in [("english", en_combined), ("arabic", ar_combined)]:
        print(f"Submitting {label} document ({file_path.name})...")
        result = client.submit_document(file_path=str(file_path))
        print(f"  Response: {result}")
        doc_id = result.get("id") or result.get("doc_id") or result.get("document_id")
        if doc_id:
            doc_ids[label] = doc_id
            print(f"  ✅ {label} doc_id: {doc_id}")
        else:
            print(f"  ⚠️  No id found in response — full response: {result}")

    DOC_ID_CACHE.write_text(json.dumps(doc_ids, indent=2))
    print(f"\nSaved doc IDs to {DOC_ID_CACHE}")

## Step 4 — Wait for Indexing to Complete

`is_retrieval_ready(doc_id)` returns True when the reasoning tree is built and queries can be submitted.

In [ ]:
## Step 5 — Inspect the Reasoning Tree

`get_tree(doc_id)` returns the full tree structure PageIndex built — the nodes it will traverse during retrieval.

In [ ]:
def wait_until_ready(doc_id: str, label: str, timeout: int = 120) -> bool:
    """Poll is_retrieval_ready() until True or timeout."""
    print(f"Waiting for '{label}' ({doc_id}) to be ready...", end="")
    for i in range(timeout):
        if client.is_retrieval_ready(doc_id):
            print(f" ✅ Ready after {i+1}s")
            return True
        print(".", end="", flush=True)
        time.sleep(1)
    print(f" ❌ Timed out after {timeout}s")
    return False

ready = {}
for label, doc_id in doc_ids.items():
    ready[label] = wait_until_ready(doc_id, label)

print(f"\nEnglish ready: {ready.get('english', False)}")
print(f"Arabic ready : {ready.get('arabic', False)}")

for label, doc_id in doc_ids.items():
    print(f"\n{'='*60}")
    print(f"REASONING TREE — {label.upper()} (doc_id: {doc_id})")
    print(f"{'='*60}")
    tree = client.get_tree(doc_id, node_summary=True)
    nodes = tree.get("nodes") or tree.get("structure") or tree.get("tree") or []
    print(f"Total nodes: {len(nodes)}")
    for node in nodes[:10]:   # show first 10 nodes
        node_id = node.get("node_id") or node.get("id", "?")
        title   = node.get("title") or node.get("name", "(no title)")
        summary = node.get("summary") or node.get("text", "")[:120]
        print(f"  [{node_id}] {title}")
        if summary:
            print(f"         {summary.replace(chr(10), ' ')}...")
    if len(nodes) > 10:
        print(f"  ... and {len(nodes)-10} more nodes")

In [ ]:
## Step 6 — Submit Queries (submit_query + chat_completions)

Two approaches available in the SDK:

## Step 6A — submit_query() — raw retrieval result

In [ ]:
en_doc_id = doc_ids.get("english")
ar_doc_id = doc_ids.get("arabic")

# English query
print("=== submit_query — ENGLISH ===")
q_en = "What are the livability indicators for Madinah in 2024?"
result_en = client.submit_query(doc_id=en_doc_id, query=q_en)
print(f"Q: {q_en}")
print(f"\nAnswer:\n{result_en.get('answer', 'No answer field')}")
print(f"\nCitations: {result_en.get('citations', [])}")
print(f"\nReasoning: {result_en.get('reasoning', '')[:300]}")

print("\n" + "="*60)

# Arabic query
print("=== submit_query — ARABIC ===")
q_ar = "ما هي مؤشرات قابلية العيش في المدينة المنورة؟"
result_ar = client.submit_query(doc_id=ar_doc_id, query=q_ar)
print(f"Q: {q_ar}")
print(f"\nAnswer:\n{result_ar.get('answer', 'No answer field')}")
print(f"\nCitations: {result_ar.get('citations', [])}")

print("\n\nFull response keys:", list(result_en.keys()))

## Step 7 — Spike Summary

## Step 6B — chat_completions() — RAG chat with citations

`chat_completions` accepts `doc_id` as a list — search across English AND Arabic simultaneously.

In [ ]:
both_doc_ids = [id for id in [en_doc_id, ar_doc_id] if id]

print("=== chat_completions — BILINGUAL (both docs) ===\n")

response = client.chat_completions(
    messages = [
        {"role": "system", "content": "You are an expert analyst of Madinah urban development reports. Cite your sources."},
        {"role": "user",   "content": "What are the main livability indicators and sustainability goals in the 2024 Madinah report?"}
    ],
    doc_id           = both_doc_ids,   # search across English + Arabic
    enable_citations = True
)

print("Answer:")
print(response.get("choices", [{}])[0].get("message", {}).get("content", response))

# Show whatever keys are returned
print(f"\nResponse keys: {list(response.keys())}")

In [ ]:
print("=" * 55)
print("SPIKE 04 — RESULTS")
print("=" * 55)
print(f"English doc_id : {doc_ids.get('english', '❌ missing')}")
print(f"Arabic doc_id  : {doc_ids.get('arabic',  '❌ missing')}")
print(f"English ready  : {ready.get('english', False)}")
print(f"Arabic ready   : {ready.get('arabic',  False)}")
print()

passed = len(doc_ids) == 2 and all(ready.values())
if passed:
    print("✅ SPIKE PASSED")
    print("   Both documents indexed and queryable.")
    print("   Next: spike_05_retrieval.ipynb — evaluate answer quality")
else:
    print("⚠️  SPIKE INCOMPLETE — check errors above")